# Hyperparameter optimisation — ConvNeXt-Small (Kaggle GPU)

Backbone chosen by the bake-off + 3-seed confirmation (`skin/results/bakeoff/`):
**ConvNeXt-Small**. EfficientNetV2-S and Swin-V2-S were tied/worse on the metrics that
held up across seeds, and ConvNeXt won the robustness axis we care about (darken).

This searches the training hyperparameters with **Optuna** (TPE + Hyperband pruning).
Two design choices come straight from the confirmation's lesson — *seed noise on the
~700-image val set is large enough to flip rankings*:

1. **Search at 1 seed** (fast, many trials), then **re-validate the top configs across 3
   seeds** before promoting anything. We never trust a single-run winner.
2. The study is **persisted to a SQLite file** in `/kaggle/working/results/`, opened with
   `load_if_exists=True`, so a study survives Kaggle's session cap and resumes on the next
   Save Version instead of restarting.

Proxy fidelity (256 px, stratified subset, short epochs) — same as the bake-off, so we
only need the *ranking* of configs; the winner gets one full-fidelity run in
`kaggle_training.ipynb`. **Run:** Add Input → HAM10000 (kmader); Accelerator → GPU T4 x2;
Run All. Outputs go to `/kaggle/working/results/` → download into `skin/results/hpo/`.

## 1. Environment

In [ ]:
!pip -q install optuna optuna-integration 2>/dev/null; import optuna; print('optuna', optuna.__version__)


In [ ]:
# Kaggle ships torch, torchvision, scikit-learn, matplotlib, pillow — nothing to install.
import os, json, time, csv, random, gc
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

IN_KAGGLE = Path("/kaggle/input").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
USE_AMP = device.type == "cuda"

# AMP helpers, tolerant of the torch.amp (>=2.3) vs torch.cuda.amp API split.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(): return _GradScaler(device.type, enabled=USE_AMP)
    def amp_ctx(): return _autocast(device.type, enabled=USE_AMP)
except Exception:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(): return _GradScaler(enabled=USE_AMP)
    def amp_ctx(): return _autocast(enabled=USE_AMP)

print(f"PyTorch {torch.__version__}  device: {device}  AMP: {USE_AMP}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    if supported and arch not in supported:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(0)} ({arch}) unsupported by this PyTorch build "
            f"(supports {supported}). Switch Settings -> Accelerator -> GPU T4 x2 and Run All.")
elif IN_KAGGLE:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again.")


## 2. Configuration

In [ ]:

MODEL_NAME = "convnext_small"   # chosen by the bake-off; the search is around THIS backbone

# --- Optuna search ----------------------------------------------------------------
N_TRIALS        = 40            # resumable: re-run the cell to add more on top
SEARCH_SEED     = 0             # single seed during search (re-validated later across seeds)
REVALIDATE_TOPK = 3             # best K configs are re-run across these seeds before promotion
REVALIDATE_SEEDS = [0, 1, 2]

# --- proxy fidelity (ranking-only, same spirit as the bake-off) -------------------
PROXY_IMG_SIZE  = 256
SUBSET_FRACTION = 0.50          # a touch higher than the bake-off's 0.40 for steadier tails
HEAD_EPOCHS     = 1             # short head warm-up
FT_EPOCHS       = 8             # fine-tune budget per trial; Hyperband prunes the duds early
BATCH_SIZE      = 32

# --- split / data (same knobs as kaggle_training.ipynb) ---------------------------
SEED = 42                       # fixes the grouped split (read by the reused split cell)
VAL_FRACTION = 0.15
VAL_CAP      = 200
TRAIN_CAP    = 100_000
NUM_WORKERS  = min(4, os.cpu_count() or 2)
VAL_RESIZE   = round(PROXY_IMG_SIZE * 256 / 224)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if IN_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/skin-cancer-mnist-ham10000")
    OUT_DIR   = Path("/kaggle/working/results")
else:
    INPUT_DIR = Path.cwd() / "data" / "ham10000_raw"
    OUT_DIR   = Path.cwd() / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB = f"sqlite:///{OUT_DIR / 'optuna_convnext.db'}"   # resumable study storage

DX_TO_CLASS = {
    "akiec": "actinic_keratoses", "bcc": "basal_cell_carcinoma",
    "bkl": "benign_keratosis-like_lesions", "df": "dermatofibroma",
    "nv": "melanocytic_nevi", "mel": "melanoma", "vasc": "vascular_lesions",
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"HPO over {MODEL_NAME}: {N_TRIALS} trials, proxy {PROXY_IMG_SIZE}px subset {SUBSET_FRACTION:.0%}")
print(f"Re-validate top {REVALIDATE_TOPK} across seeds {REVALIDATE_SEEDS}")
print(f"Study DB: {STUDY_DB}")


## 3. Lesion-grouped split + stratified subset (reused from the bake-off)

In [ ]:
# Locate dataset (slug/layout can vary) — find HAM10000_metadata.csv under /kaggle/input.
search_roots = [INPUT_DIR] + ([Path("/kaggle/input")] if IN_KAGGLE else [])
meta_csv = None
for root in search_roots:
    if root.exists():
        meta_csv = next(root.rglob("HAM10000_metadata*.csv"), None)
        if meta_csv is not None:
            break
if meta_csv is None:
    raise FileNotFoundError(
        "Could not find HAM10000_metadata.csv. Add the dataset via Add Input -> "
        "'Skin Cancer MNIST: HAM10000' (by kmader).")
INPUT_DIR = meta_csv.parent
print(f"Dataset root: {INPUT_DIR}")

img_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    for p in INPUT_DIR.rglob(ext):
        img_paths.setdefault(p.stem, p)
assert img_paths, f"No image files under {INPUT_DIR}."

with open(meta_csv, newline="") as f:
    rows = list(csv.DictReader(f))

lesion_imgs, lesion_dx = defaultdict(list), {}
for r in rows:
    iid, lid, dx = r["image_id"], r["lesion_id"], r["dx"].lower()
    if iid not in img_paths:
        continue
    lesion_imgs[lid].append(iid)
    lesion_dx[lid] = dx

classes = sorted(DX_TO_CLASS[dx] for dx in set(lesion_dx.values()))
class_to_idx = {c: i for i, c in enumerate(classes)}
assert len(classes) == 7, f"Expected 7 classes, got {classes}"

lesions_by_class = defaultdict(list)
for lid, dx in lesion_dx.items():
    lesions_by_class[DX_TO_CLASS[dx]].append(lid)

# Per-class grouped split: a lesion's images never straddle train/val.
rng = random.Random(SEED)
train_samples, val_samples = [], []
for cls in classes:
    lesions = lesions_by_class[cls]
    rng.shuffle(lesions)
    total = sum(len(lesion_imgs[l]) for l in lesions)
    val_target = min(VAL_CAP, max(1, round(VAL_FRACTION * total)))
    val_n, train_n = 0, 0
    for lid in lesions:
        imgs = [(img_paths[iid], class_to_idx[cls]) for iid in lesion_imgs[lid]]
        if val_n < val_target:
            val_samples.extend(imgs); val_n += len(imgs)
        elif train_n < TRAIN_CAP:
            take = imgs[: max(0, TRAIN_CAP - train_n)]
            train_samples.extend(take); train_n += len(take)

# Stratified subsample of TRAIN: identical fraction from every class, so the class
# prior is unchanged -> the logit-adjustment offset stays valid at full scale.
by_cls = defaultdict(list)
for s in train_samples:
    by_cls[s[1]].append(s)
srng = random.Random(SEED)
subset = []
for ci, items in by_cls.items():
    srng.shuffle(items)
    subset.extend(items[: max(1, round(SUBSET_FRACTION * len(items)))])
srng.shuffle(subset)
train_samples = subset

print(f"Train (subset): {len(train_samples)}   Val: {len(val_samples)}")
for cls in classes:
    ci = class_to_idx[cls]
    tr = sum(1 for _, y in train_samples if y == ci)
    va = sum(1 for _, y in val_samples if y == ci)
    print(f"  {cls:30s} train {tr:5d}  val {va:4d}")


## 4. Data, head-swap, and metric helpers (reused from the bake-off)

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, samples, transform, classes):
        self.samples, self.transform, self.classes = samples, transform, classes
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(PROXY_IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_ds = SkinDataset(train_samples, train_tf, classes)
_pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=_pin,
                          persistent_workers=NUM_WORKERS > 0, drop_last=True)


def make_val_loader(perturb=None):
    # Base pipeline -> [0,1] CHW tensor; a perturbation acts on that, then normalise.
    steps = [transforms.Resize(VAL_RESIZE), transforms.CenterCrop(PROXY_IMG_SIZE),
             transforms.ToTensor()]
    if perturb is not None:
        steps.append(transforms.Lambda(perturb))
    steps.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    ds = SkinDataset(val_samples, transforms.Compose(steps), classes)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=_pin)

val_loader = make_val_loader()  # clean, used for epoch selection


def replace_head(model, n_classes):
    for attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, attr, None)
        if head is None:
            continue
        if isinstance(head, nn.Linear):
            setattr(model, attr, nn.Linear(head.in_features, n_classes)); return model
        last = None
        for name, m in head.named_modules():
            if isinstance(m, nn.Linear):
                last = name
        if last is not None:
            parent = head; *path, leaf = last.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, leaf, nn.Linear(getattr(parent, leaf).in_features, n_classes))
            return model
    raise ValueError(f"No head on {type(model).__name__}")


def freeze_backbone(model, freeze=True):
    head_ids = set()
    for attr in ["classifier", "head", "heads", "fc"]:
        h = getattr(model, attr, None)
        if h is not None:
            head_ids.update(id(p) for p in h.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_ids) if freeze else True


def head_parameters(model):
    for attr in ["classifier", "head", "heads", "fc"]:
        h = getattr(model, attr, None)
        if h is not None:
            return h.parameters()
    return model.parameters()


def log_class_prior(samples, n_classes):
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    prior = counts / counts.sum()
    return torch.log(torch.tensor(prior, dtype=torch.float32).clamp_min(1e-12))


class LogitAdjustedLoss(nn.Module):
    def __init__(self, log_prior, tau):
        super().__init__(); self.register_buffer("adj", tau * log_prior)
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target)


def normalized_entropy(p):
    p = np.asarray(p, dtype=float); n = len(p)
    return 0.0 if n <= 1 else float(-(p * np.log(p + 1e-12)).sum() / np.log(n))


def expected_calibration_error(probs, targets, n_bins=15):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == targets).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi); c = int(m.sum())
        if c:
            ece += c / len(conf) * abs(conf[m].mean() - acc[m].mean())
    return float(ece)


# Perturbations act on a [0,1] CHW tensor (same defs as 04_analysis.py); we evaluate
# each at its worst severity from the analysis sweep.
def gaussian_noise(std): return lambda t: (t + torch.randn_like(t) * std).clamp(0, 1)
def gaussian_blur(sigma):
    k = 2 * int(np.ceil(2 * sigma)) + 1
    return lambda t: TF.gaussian_blur(t, kernel_size=k, sigma=sigma)
def brightness(factor): return lambda t: TF.adjust_brightness(t, factor).clamp(0, 1)

WORST = {"Darken": brightness(0.25), "Gaussian noise": gaussian_noise(0.30),
         "Blur": gaussian_blur(3.5)}


## 5. Search space and the trainable objective

Six knobs, deliberately tight. Note two are problem-specific:
- **`tau`** — the logit-adjustment strength (imbalance corrector).
- **`aug_strength`** — brightness/contrast jitter, the documented fix for the *darken*
  failure; letting the search tune it means it can buy robustness, not just accuracy.

In [ ]:

class LogitAdjustedLossLS(nn.Module):
    """Logit adjustment (Menon 2021) with optional label smoothing."""
    def __init__(self, log_prior, tau, label_smoothing=0.0):
        super().__init__()
        self.register_buffer("adj", tau * log_prior)
        self.ls = label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target, label_smoothing=self.ls)


def make_train_loader(aug_strength, seed):
    tf = transforms.Compose([
        transforms.RandomResizedCrop(PROXY_IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=aug_strength, contrast=aug_strength,
                               saturation=aug_strength * 0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    ds = SkinDataset(train_samples, tf, classes)
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=_pin, persistent_workers=False, drop_last=True, generator=g)


def run_inference(model, loader):
    model.eval(); P, PR, T = [], [], []
    with torch.no_grad(), amp_ctx():
        for x, y in loader:
            logits = model(x.to(device, non_blocking=True))
            p = torch.softmax(logits.float(), 1).cpu().numpy()
            P.append(p); PR.append(p.argmax(1)); T.append(y.numpy())
    return np.concatenate(P), np.concatenate(PR), np.concatenate(T)


def build(hp, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = replace_head(models.get_model(MODEL_NAME, weights="DEFAULT"), len(classes)).to(device)
    crit = LogitAdjustedLossLS(log_class_prior(train_samples, len(classes)),
                               hp["tau"], hp["label_smoothing"]).to(device)
    loader = make_train_loader(hp["aug_strength"], seed)
    return model, crit, loader


def train_head(model, crit, loader, scaler, hp):
    freeze_backbone(model, True)
    opt = torch.optim.AdamW(head_parameters(model), lr=hp["head_lr"], weight_decay=hp["weight_decay"])
    for _ in range(HEAD_EPOCHS):
        model.train()
        for x, y in loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            opt.zero_grad()
            with amp_ctx():
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()


def ft_epoch(model, crit, opt, loader, scaler):
    model.train()
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with amp_ctx():
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()


def suggest(trial):
    return {
        "head_lr":         trial.suggest_float("head_lr", 3e-4, 3e-3, log=True),
        "ft_lr":           trial.suggest_float("ft_lr", 1e-5, 3e-4, log=True),
        "weight_decay":    trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True),
        "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.2),
        "tau":             trial.suggest_float("tau", 0.5, 1.5),
        "aug_strength":    trial.suggest_float("aug_strength", 0.1, 0.5),
    }


def objective(trial):
    hp = suggest(trial)
    model, crit, loader = build(hp, SEARCH_SEED)
    scaler = make_scaler()
    train_head(model, crit, loader, scaler, hp)
    freeze_backbone(model, False)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["ft_lr"], weight_decay=hp["weight_decay"])
    best = -1.0
    for epoch in range(FT_EPOCHS):
        ft_epoch(model, crit, opt, loader, scaler)
        _, pr, t = run_inference(model, val_loader)
        bal = balanced_accuracy_score(t, pr)
        best = max(best, bal)
        trial.report(bal, step=epoch)         # Hyperband sees per-epoch progress...
        if trial.should_prune():               # ...and kills unpromising trials early.
            del model; gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            raise optuna.TrialPruned()
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return best


## 6. Run the search (resumable)

In [ ]:

study = optuna.create_study(
    study_name="convnext_hpo", direction="maximize",
    storage=STUDY_DB, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEARCH_SEED),
    pruner=optuna.pruners.HyperbandPruner(min_resource=2, max_resource=FT_EPOCHS),
)
t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
done = [t for t in study.trials if t.state.name == "COMPLETE"]
pruned = [t for t in study.trials if t.state.name == "PRUNED"]
print(f"\n{len(study.trials)} trials ({len(done)} complete, {len(pruned)} pruned) "
      f"in {(time.time()-t0)/60:.1f} min")
print(f"Best single-seed val bal-acc: {study.best_value:.3f}")
print("Best params:", json.dumps(study.best_params, indent=2))


## 7. Re-validate the top configs across seeds, then promote

The seed-noise lesson in action: instead of crowning the single-seed `best_trial`, we
re-run the **top K** configs across several seeds and rank by **mean** balanced accuracy,
breaking ties (within ~1 std) on ECE then darken robustness — exactly the criteria the
3-seed confirmation taught us to use.

In [ ]:

WORST_DARKEN = brightness(0.25)

def full_measure(hp, seed):
    model, crit, loader = build(hp, seed)
    scaler = make_scaler()
    train_head(model, crit, loader, scaler, hp)
    freeze_backbone(model, False)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["ft_lr"], weight_decay=hp["weight_decay"])
    best, best_state = -1.0, None
    for _ in range(FT_EPOCHS):
        ft_epoch(model, crit, opt, loader, scaler)
        _, pr, t = run_inference(model, val_loader)
        bal = balanced_accuracy_score(t, pr)
        if bal > best:
            best = bal
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    probs, preds, targets = run_inference(model, val_loader)
    dp, dpr, dt = run_inference(model, make_val_loader(WORST_DARKEN))
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return {"bal_acc": float(balanced_accuracy_score(targets, preds)),
            "ece": expected_calibration_error(probs, targets),
            "darken_bal_acc": float(balanced_accuracy_score(dt, dpr))}

import statistics as st
top = sorted([t for t in study.trials if t.state.name == "COMPLETE"],
             key=lambda t: t.value, reverse=True)[:REVALIDATE_TOPK]
revalidated = []
for rank, t in enumerate(top):
    seeds_m = [full_measure(t.params, s) for s in REVALIDATE_SEEDS]
    agg = {k: (st.mean([m[k] for m in seeds_m]),
               st.pstdev([m[k] for m in seeds_m]) if len(seeds_m) > 1 else 0.0)
           for k in seeds_m[0]}
    revalidated.append({"trial": t.number, "params": t.params,
                        "search_bal_acc": t.value, "agg": agg})
    print(f"trial {t.number}: bal-acc {agg['bal_acc'][0]:.3f}±{agg['bal_acc'][1]:.3f}  "
          f"ECE {agg['ece'][0]:.3f}  darken {agg['darken_bal_acc'][0]:.3f}", flush=True)

# Promote: best mean bal-acc; ties (within 1 std) broken on ECE then darken robustness.
best_ba = max(r["agg"]["bal_acc"][0] for r in revalidated)
tied = [r for r in revalidated
        if best_ba - r["agg"]["bal_acc"][0] <= r["agg"]["bal_acc"][1] + 1e-9]
winner = min(tied, key=lambda r: (r["agg"]["ece"][0], -r["agg"]["darken_bal_acc"][0]))
print(f"\n>>> PROMOTE trial {winner['trial']} "
      f"(bal-acc {winner['agg']['bal_acc'][0]:.3f}, ECE {winner['agg']['ece'][0]:.3f}, "
      f"darken {winner['agg']['darken_bal_acc'][0]:.3f})")
print("Best hyperparameters:", json.dumps(winner["params"], indent=2))

with open(OUT_DIR / "best_params.json", "w") as f:
    json.dump({"model": MODEL_NAME, "params": winner["params"],
               "revalidated_metrics": winner["agg"]}, f, indent=2)
with open(OUT_DIR / "hpo_results.json", "w") as f:
    json.dump({"model": MODEL_NAME, "n_trials": len(study.trials),
               "search_best_value": study.best_value, "search_best_params": study.best_params,
               "revalidated_topk": revalidated, "winner_trial": winner["trial"]}, f, indent=2)
print(f"\nSaved best_params.json + hpo_results.json to {OUT_DIR}")
print("Next: set these HPs + MODEL_NAME=convnext_small in kaggle_training.ipynb, run the full recipe.")


## 8. Diagnostic plots

In [ ]:

try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    fig = plot_optimization_history(study); fig.figure.tight_layout()
    fig.figure.savefig(OUT_DIR / "hpo_history.png", dpi=130, bbox_inches="tight")
    if len([t for t in study.trials if t.state.name == "COMPLETE"]) > 1:
        fig2 = plot_param_importances(study); fig2.figure.tight_layout()
        fig2.figure.savefig(OUT_DIR / "hpo_importances.png", dpi=130, bbox_inches="tight")
    print("Saved hpo_history.png + hpo_importances.png")
except Exception as e:
    print("Plot step skipped:", e)
